[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://drive.google.com/file/d/1jmSWbICc0lY8tjO6XWkDrTqcYBN-MP_r/view?usp=sharing)

# LLM Evaluation – Partial Dataset

This notebook demonstrates how to evaluate when you have only questions (no answers yet). Floeval generates `llm_response` for each sample at runtime, then runs the metrics on the generated responses.

**Objectives**
- Install Floeval and configure credentials
- Load a partial dataset from a JSON file you provide
- Set `dataset_generator_model` so Floeval generates responses
- Run evaluation and inspect results

## 1. Installation

Install Floeval before running this notebook.

In [ ]:
%pip install floeval

## 2. Configuration Constants

Set the following constants before running. Replace placeholder values with your API credentials and model identifiers.

**Provider flexibility:** You can use any OpenAI-compatible provider (OpenAI, Azure OpenAI, Anthropic, local models, etc.) — set the appropriate `base_url` and model names for your provider.

**Using FloTorch:** If you want to use FloTorch keys and gateway, obtain credentials from the [FloTorch Console](https://docs.flotorch.cloud/introduction/).

In [ ]:
import getpass
# LLM and API configuration

OPENAI_BASE_URL = "https://api.openai.com/v1"
OPENAI_API_KEY = getpass.getpass("your-api-key")
OPENAI_CHAT_MODEL = "gpt-4o-mini"
OPENAI_EMBEDDING_MODEL = "text-embedding-3-small"

## 3. Imports

Import evaluation components and the LLM configuration schema.

In [ ]:
from pathlib import Path

from floeval import DatasetLoader, Evaluation
from floeval.config.schemas.io.llm import OpenAIProviderConfig


## 4. Configure the LLM

The LLM configuration is built using the constants defined above. It is used for both response generation and evaluation.

In [ ]:
llm_config = OpenAIProviderConfig(
    base_url=OPENAI_BASE_URL,
    api_key=OPENAI_API_KEY,
    chat_model=OPENAI_CHAT_MODEL,
    embedding_model=OPENAI_EMBEDDING_MODEL,
)

## 5. Load the Partial Dataset

Minimal shapes (no `llm_response` yet; Floeval generates it):

JSON with `samples` array — `{ "samples": [ { "user_input": "..." } ] }`  
JSONL — one object per line: `{ "user_input": "...", "ground_truth": "..." }` (optional fields allowed).

**Example file** — partial questions (no `llm_response`).  
<a href="../datasets/llm_evaluation/partial_dataset_squad_top50.jsonl" download="partial_dataset_squad_top50.jsonl">partial_dataset_squad_top50.jsonl</a>

Provide the partial dataset `.jsonl` or `.json` path (or upload in Colab), then load it with `DatasetLoader`.


In [ ]:
try:
    from google.colab import files

    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

if _IN_COLAB:
    print("Upload your dataset .jsonl/.json file:")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded.")
    dataset_path = Path(next(iter(uploaded.keys())))
else:
    dataset_path = (
        Path(input("Enter path to dataset .jsonl/.json file: ").strip().strip('"'))
        .expanduser()
        .resolve()
    )
    if not dataset_path.exists():
        raise FileNotFoundError(f"File not found: {dataset_path}")
    else:
        print("✅ Dataset file found.")


### Resolve Dataset Path

Provide `dataset_path` via file upload in Colab or local `.jsonl`/`.json` path input in Jupyter.


In [ ]:
partial_dataset = DatasetLoader.from_file(dataset_path, partial_dataset=True)
print(
    f"Partial dataset loaded from {dataset_path}: {len(partial_dataset.samples)} samples (no llm_response)"
)


### Load Partial Dataset

Load and validate the partial dataset with `DatasetLoader.from_file(..., partial_dataset=True)`.


## 6. Create and Run the Evaluation

Set `dataset_generator_model` to generate responses from `user_input`, then run async evaluation.

In [ ]:
evaluation = Evaluation(
    dataset=partial_dataset,
    llm_config=llm_config,
    metrics=["answer_relevancy"],
    default_provider="ragas",
    dataset_generator_model=OPENAI_CHAT_MODEL,
)


### Build Evaluation Object

Configure `Evaluation` with partial dataset generation and metric settings.


In [ ]:
results = await evaluation.arun()
print("Aggregate scores:", results.aggregate_scores)


### Run Evaluation (Async)

Execute `await evaluation.arun()` to compute and print aggregate metric scores.


## 7. Inspect Generated Responses

Each sample result includes the generated `llm_response` and the metric scores. This enables inspection of both the generated text and the evaluation scores per sample.

### Inspect per-sample results

Iterates `results.sample_results` to print each question snippet and metric scores.


In [ ]:
for i, sr in enumerate(results.sample_results, start=1):
    print(f"Sample {i}: {sr['user_input']}")
    print(f"  Generated: {sr.get('llm_response', '')[:80]}...")
    for key, data in sr.get("metrics", {}).items():
        print(f"  {key}: {data.get('score')}")

## Summary

This notebook demonstrated the evaluation of LLM responses using a partial dataset with Floeval.

The key components included:

1. **Partial Dataset Loading**: A dataset with `user_input` only was loaded from `.jsonl`/`.json` using `DatasetLoader.from_file(..., partial_dataset=True)`.
2. **LLM Configuration**: The OpenAI-compatible provider was configured for both generation and evaluation.
3. **Response Generation**: The `dataset_generator_model` parameter enabled Floeval to generate responses at runtime before scoring.
4. **Evaluation Execution**: The `answer_relevancy` metric was run on the generated responses.
5. **Results Inspection**: Generated text and per-sample scores were accessed through `results.sample_results`.

This example showcases the workflow for evaluating when you have questions but no pre-generated answers.